# 05d Run FaIR REN

In [1]:
import os
import json
import copy
import pyam
import pandas as pd

from climate_assessment.climate.fair import get_fair_configurations
from climate_assessment.climate import clean_wg3_scenarios

from scmdata import ScmRun
from openscm_runner.run import run
from openscm_runner.adapters import FAIR
from openscm_runner.utils import calculate_quantiles

import matplotlib.pyplot as plt
import seaborn as sns

<IPython.core.display.Javascript object>

/Users/gauravganti/opt/anaconda3/envs/cdr_climate_uncertainty/lib/python3.10/site-packages/scmdata/database/_database.py:9: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  import tqdm.autonotebook as tqdman


Step 1: Load the emissions data file.

In [2]:
df_emissions = ScmRun(
    os.path.join(
        '..',
        '..',
        'Data',
        'pathways',
        'ar6_imp_rensp',
        'ar6_ren_sp_emissions_for_climate_assessment.csv'
    ),
    lowercase_cols=True
)

Step 2: Load the FaIR configurations.

In [3]:
data_dir = os.path.join('..','..','Data', 'fair')
fair_slim_filename = 'fair-1.6.2-wg3-params-slim.json'
fair_common_filename = 'fair-1.6.2-wg3-params-common.json'

In [4]:
fair_config = get_fair_configurations(
    fair_version='1.6.2',
    fair_probabilistic_file=os.path.join(data_dir, fair_slim_filename),
    fair_extra_config=os.path.join(data_dir, fair_common_filename),
    num_cfgs=2237
)

Step 3: Trim length of config.

In [5]:
nt = df_emissions.time_points.years()[-1] - 1750 + 1
nt

361

In [6]:
updated_config = copy.copy(fair_config)
for i in range(len(fair_config)):
    updated_config[i]['F_solar'] = updated_config[i]['F_solar'][:nt]
    updated_config[i]['F_volcanic'] = updated_config[i]['F_volcanic'][:nt]
    updated_config[i]['natural'] = updated_config[i]['natural'][:nt]

Step 4: Run FaIR

In [7]:
temp = run(
    climate_models_cfgs={
        'FAIR':updated_config
    },
    scenarios=df_emissions,
    output_variables=(
        "Surface Air Temperature Change",
    ),
)

Climate models:   0%|          | 0.00/1.00 [00:00<?, ?it/s]

Creating FaIR emissions inputs:   0%|          | 0.00/2.00 [00:00<?, ?it/s]

/Users/gauravganti/opt/anaconda3/envs/cdr_climate_uncertainty/lib/python3.10/site-packages/openscm_units/_unit_registry.py:471: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for label, val in metric_conversion.iteritems():


Front serial:   0%|          | 0.00/3.00 [00:00<?, ?it/s]

Front parallel:   0%|          | 0.00/2.00 [00:00<?, ?it/s]

Parallel runs:   0%|          | 0.00/4.47k [00:00<?, ?it/s]

Step 5: Rebase the output

In [8]:
temp_rebased = (
    temp
    .relative_to_ref_period_mean(
        year=range(
            1850,
            1901
        )
    )
)

Step 6: Calculate the required quantiles

In [9]:
necessary_quantiles = [
    0.1,
    0.167,
    0.2,
    0.3,
    0.33,
    0.4,
    0.5,
    0.6,
    0.66,
    0.7,
    0.8,
    0.833,
    0.9
]

In [10]:
temp_rebased_quantiles = calculate_quantiles(
    temp_rebased, 
    necessary_quantiles
)

In [11]:
temp_rebased_quantiles_pyam = (
    temp_rebased_quantiles
    .to_iamdataframe()
    .swap_time_for_year()
    .filter(year=range(1750, 2101))
)

Step 7: Write this out

In [12]:
temp_rebased_quantiles_pyam.to_csv(
    os.path.join(
        '..',
        '..',
        'Data',
        'pathways',
        'ar6_imp_rensp',
        'ar6_ren_sp_climate_assessed.csv'
    )
)